# Proyecto 2 — Modelo B: U-Net 3D temporal + transformer

**CC3084 Data Science · UVG · Semestre II 2026**
Isabella Obando · Anthony Lou · Fabian Prado · Fernando Mendoza

Modelo base publicado por los organizadores de la competencia (Royer Lab, 2026), que la investigación
del grupo seleccionó como alternativa con aprendizaje (secciones 3.1 y 3.2):

| Etapa | Método |
|---|---|
| Detección | **U-Net 3D con atención temporal**: produce un mapa de probabilidad de centro de núcleo, y los centros son sus máximos locales (umbral 0.99, promediando volteos en XY) |
| Enlace | un **transformer de atención cruzada** puntúa cada par de nodos (t, t+1) con los rasgos de la U-Net; después, una **optimización lineal entera (ILP)** elige enlaces globalmente consistentes |
| Supervisión | solo las aristas anotadas: las células sin anotar no entran en la pérdida |

Se usan los pesos públicos, entrenados 3 épocas con el *split 0* (90% de las muestras de `train/`).

**Qué hace este notebook:**

1. **Validación:** corre el modelo en las 19 muestras del *split 0* que **no** se usaron para
   entrenar y las evalúa con la métrica oficial (`evaluate.py` del repositorio de la competencia), en
   total y por embrión, junto con el tiempo de cómputo.
2. **Envío:** corre el modelo sobre `test/` y genera `submission.csv`. En el *rerun* sobre el
   conjunto oculto, la validación se salta.

**Cómo correrlo en Kaggle:**

- Accelerator **GPU T4 ×2**, **Internet Off**.
- Como *Input*, agregar la competencia **y** el dataset **`cellmot-baseline-artifacts`** (de
  *thibautgoldsborough*: + Add Input → buscarlo por nombre). Trae los pesos, el código y las librerías
  para instalar sin internet.

In [ ]:
import os, io, sys, json, time, random, shutil, subprocess, contextlib
from pathlib import Path

import numpy as np
import pandas as pd

COMPETITION = "biohub-cell-tracking-during-development"
ES_RERUN = bool(os.environ.get("KAGGLE_IS_COMPETITION_RERUN"))
VALIDAR = not ES_RERUN
ENTRADA = Path("/kaggle/input")


def buscar_carpeta(raiz, condicion, niveles=5):
    """Primera carpeta bajo raiz (hasta `niveles` de profundidad) que cumple la condición."""
    for prof in range(1, niveles + 1):
        for p in sorted(raiz.glob("/".join(["*"] * prof))):
            if p.is_dir() and condicion(p):
                return p
    return None


def contenido_entrada():
    return sorted(str(p.relative_to(ENTRADA)) for p in ENTRADA.glob("*/*")) if ENTRADA.exists() else []


def encontrar_datos():
    if os.environ.get("BIOHUB_DATA"):
        return Path(os.environ["BIOHUB_DATA"])
    for c in [ENTRADA / "competitions" / COMPETITION, ENTRADA / COMPETITION]:
        if (c / "test").is_dir():
            return c
    c = buscar_carpeta(ENTRADA, lambda p: (p / "test").is_dir() and any((p / "test").glob("*.zarr")), 4)
    if c is None:
        raise FileNotFoundError(
            "No se encontró la competencia. + Add Input -> Competitions -> 'Biohub - Cell Tracking During "
            f"Development'.\nContenido de {ENTRADA}: {contenido_entrada() or 'vacío'}")
    return c


def encontrar_artefactos():
    if os.environ.get("BIOHUB_ARTIFACTS"):
        return Path(os.environ["BIOHUB_ARTIFACTS"])
    c = buscar_carpeta(ENTRADA, lambda p: (p / "wheels").is_dir() and (p / "weights").is_dir())
    if c is None:
        raise FileNotFoundError(
            "No se encontró el dataset 'cellmot-baseline-artifacts' (pesos y librerías del modelo B).\n"
            "En el panel derecho: + Add Input -> Datasets -> buscar 'cellmot-baseline-artifacts' "
            "(autor: thibautgoldsborough) -> Add.\n"
            f"Contenido actual de {ENTRADA}: {contenido_entrada() or 'vacío'}")
    return c


DATA_DIR = encontrar_datos()
TRAIN_DIR, TEST_DIR = DATA_DIR / "train", DATA_DIR / "test"
ARTIFACTS = encontrar_artefactos()
TRABAJO = Path(os.environ.get("BIOHUB_SALIDA", "/kaggle/working"))
REPO_DIR = TRABAJO / "repo"
METODO = "unet_transformer"
PESOS = f"weights/{METODO}/split_0/edge_predictor_best.pth"

# Configuración de la notebook oficial de los organizadores (puntaje oficial 0.812)
B_CFG = dict(det_threshold=0.99, unet_batch_size=4, use_ilp=True, ilp_edge_weight=-1.0,
             ilp_appearance_weight=0.1, ilp_disappearance_weight=0.1, ilp_division_weight=1.0)

print(f"Datos: {DATA_DIR}\nArtefactos: {ARTIFACTS}\nRerun: {ES_RERUN}")

## Instalación sin internet

Igual que en la notebook oficial: las librerías se instalan desde los *wheels* del dataset de
artefactos, y el código del repositorio y los pesos se copian a una carpeta con permiso de escritura.

In [ ]:
# Los wheels del dataset de artefactos están compilados para una versión concreta de Python (3.12).
# Si el entorno de Kaggle usa otra, la instalación sin internet falla: se avisa antes de intentarla.
tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
incompatibles = [w.name for w in (ARTIFACTS / "wheels").glob("*.whl")
                 if "-cp3" in w.name and "abi3" not in w.name and f"-{tag}-" not in w.name]
if incompatibles:
    raise RuntimeError(
        f"Este entorno usa Python {sys.version.split()[0]}, pero los wheels del modelo B son para otra versión "
        f"(p. ej. {incompatibles[0]}).\nSolución: abrir la notebook oficial 'UNet baseline - inference & submission' "
        "(thibautgoldsborough) -> Copy & Edit; en la copia, File -> Import Notebook con este archivo. La copia "
        "conserva el entorno original (Python 3.12) y los Inputs.")
print(f"Python {sys.version.split()[0]}: wheels compatibles")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index", "--find-links",
                str(ARTIFACTS / "wheels"), "tracksdata", "zarr>=3.0.10", "pyscipopt"], check=True)
shutil.copytree(ARTIFACTS / "repo", REPO_DIR, dirs_exist_ok=True)
shutil.copytree(ARTIFACTS / "weights", REPO_DIR / "weights", dirs_exist_ok=True)
for sub in ("src", "scripts"):
    sys.path.insert(0, str(REPO_DIR / sub))
print("Pesos:", os.listdir(REPO_DIR / "weights" / METODO / "split_0"))


def correr_modelo_b(data_dir, nombres, destino):
    """Inferencia con el script oficial sobre `nombres`; copia los .geff a `destino`. Devuelve segundos."""
    splits = REPO_DIR / "splits_ejecucion.json"
    splits.write_text(json.dumps([{"split": 0, "train": [], "test": list(nombres)}]))
    cmd = [sys.executable, "scripts/predict_unet_transformer.py",
           "--data-dir", str(data_dir), "--splits", str(splits), "--split", "0", "--weights", PESOS,
           "--unet-batch-size", str(B_CFG["unet_batch_size"]), "--det-threshold", str(B_CFG["det_threshold"]),
           "--ilp-edge-weight", str(B_CFG["ilp_edge_weight"]),
           "--ilp-appearance-weight", str(B_CFG["ilp_appearance_weight"]),
           "--ilp-disappearance-weight", str(B_CFG["ilp_disappearance_weight"]),
           "--ilp-division-weight", str(B_CFG["ilp_division_weight"])]
    if B_CFG["use_ilp"]:
        cmd.append("--use-ilp")
    t0 = time.time()
    subprocess.run(cmd, cwd=REPO_DIR, env={**os.environ, "PYTHONPATH": "src"}, check=True)
    seg = time.time() - t0
    destino = Path(destino)
    shutil.rmtree(destino, ignore_errors=True)
    destino.mkdir(parents=True)
    for g in REPO_DIR.glob(f"predictions/*/{METODO}/split_0/*.geff"):
        (shutil.copytree if g.is_dir() else shutil.copy)(g, destino / g.name)
    return seg


def leer_geff_prediccion(ruta):
    """Grafo .geff predicho -> (nodos node_id, t, z, y, x; aristas source_id, target_id)."""
    import tracksdata as td
    g = td.graph.IndexedRXGraph.from_geff(ruta)
    g = g[0] if isinstance(g, tuple) else g
    if g.num_nodes() == 0:      # un .geff vacío pierde las columnas z, y, x al leerse
        return (pd.DataFrame({c: pd.Series(dtype=float) for c in ["node_id", "t", "z", "y", "x"]}),
                pd.DataFrame({c: pd.Series(dtype=np.int64) for c in ["source_id", "target_id"]}))
    n = g.node_attrs().to_pandas()
    e = g.edge_attrs().to_pandas()
    nodos = n[["node_id", "t", "z", "y", "x"]].astype({"node_id": np.int64, "t": int})
    aristas = (e[["source_id", "target_id"]].astype(np.int64) if len(e)
               else pd.DataFrame({c: pd.Series(dtype=np.int64) for c in ["source_id", "target_id"]}))
    return nodos, aristas

## Validación: muestras que el modelo no vio al entrenar

El script de entrenamiento oficial arma el *split 0* así: ordena las muestras que tienen `.zarr` y
`.geff`, las mezcla con `random.Random(0)` y deja el primer 10% para validación. Si el repositorio de
artefactos trae un archivo de *splits*, se usa ese. Evaluar en las muestras de entrenamiento inflaría el
puntaje.

In [ ]:
def muestras_split0():
    for f in sorted(REPO_DIR.rglob("*split*.json")):
        try:
            folds = json.loads(f.read_text())
            if isinstance(folds, list) and folds and folds[0].get("test") and folds[0].get("train"):
                print(f"Split tomado del archivo {f.relative_to(REPO_DIR)}")
                return sorted(Path(s).name for s in folds[0]["test"])
        except Exception:
            pass
    stems = sorted(p.name[:-5] for p in TRAIN_DIR.glob("*.zarr") if (TRAIN_DIR / f"{p.name[:-5]}.geff").exists())
    random.Random(0).shuffle(stems)
    n_val = max(1, len(stems) // 10)
    print(f"Split reconstruido con random.Random(0): {len(stems) - n_val} de entrenamiento / {n_val} de validación")
    return sorted(stems[:n_val])


if VALIDAR:
    VAL = muestras_split0()
    print(f"Muestras de validación: {len(VAL)} | por embrión:",
          pd.Series([s.split("_")[0] for s in VAL]).value_counts().to_dict())
    PRED_VAL = TRABAJO / "pred_val"
    SEG_VAL = correr_modelo_b(TRAIN_DIR, VAL, PRED_VAL)
    print(f"\nInferencia: {SEG_VAL:.0f} s en total, {SEG_VAL / len(VAL):.0f} s por muestra (GPU)")

In [ ]:
def evaluar_oficial(pred_dir, nombres):
    """Métrica oficial (evaluate_pairs + summarise), en total y por embrión."""
    from evaluate import evaluate_pairs
    from tracking_cellmot.metrics import summarise
    filas = {}
    grupos = [("total", nombres)] + [(e, [s for s in nombres if s.startswith(e)])
                                     for e in sorted({s.split("_")[0] for s in nombres})]
    for grupo, ss in grupos:
        tmp = TRABAJO / "eval_tmp" / grupo
        shutil.rmtree(tmp, ignore_errors=True)
        tmp.mkdir(parents=True)
        for s in ss:
            os.symlink(Path(pred_dir) / f"{s}.geff", tmp / f"{s}.geff")
        with contextlib.redirect_stdout(io.StringIO()):
            rows, _ = evaluate_pairs(tmp, TRAIN_DIR)
        r = summarise(rows)
        filas[grupo] = {"muestras": len(ss), "puntaje oficial": r["score"],
                        "Jaccard aristas ajustado": r["adj_edge_jaccard"], "Jaccard aristas": r["edge_jaccard"],
                        "Jaccard divisiones": r["division_jaccard"],
                        "divisiones TP/FP/FN": f"{r['division_tp']}/{r['division_fp']}/{r['division_fn']}",
                        "recall de nodos": r["node_recall"]}
    return pd.DataFrame(filas).T


if VALIDAR:
    RESULTADO = evaluar_oficial(PRED_VAL, VAL)
    print("Métrica oficial del modelo B (puntaje = Jaccard de aristas ajustado + 0.1 × Jaccard de divisiones):")
    display(RESULTADO)

    n_pred = sum(len(leer_geff_prediccion(PRED_VAL / f"{s}.geff")[0]) for s in VAL)
    print(f"Nodos predichos: {n_pred:,} | segundos por muestra: {SEG_VAL / len(VAL):.0f} (GPU)")

### Lectura de los resultados

> ⏳ **Interpretación pendiente**: completar con los resultados de la corrida en Kaggle.

## Envío

Se corre el modelo sobre `test/` y se escribe `submission.csv` con el formato de la competencia. En el
*rerun*, `test/` es el conjunto oculto.

In [ ]:
pruebas = sorted(p.name[:-5] for p in TEST_DIR.glob("*.zarr"))
seg = correr_modelo_b(TEST_DIR, pruebas, TRABAJO / "pred_test")
print(f"{len(pruebas)} muestras de prueba en {seg:.0f} s")

filas = []
for s in pruebas:
    nodos, aristas = leer_geff_prediccion(TRABAJO / "pred_test" / f"{s}.geff")
    filas.append(pd.DataFrame({"dataset": s, "row_type": "node", "node_id": nodos["node_id"],
                               "t": nodos["t"].astype(int), "z": nodos["z"].round().astype(int),
                               "y": nodos["y"].round().astype(int), "x": nodos["x"].round().astype(int),
                               "source_id": -1, "target_id": -1}))
    filas.append(pd.DataFrame({"dataset": s, "row_type": "edge", "node_id": -1, "t": -1, "z": -1, "y": -1,
                               "x": -1, "source_id": aristas["source_id"], "target_id": aristas["target_id"]}))
envio = pd.concat(filas, ignore_index=True)
envio.index.name = "id"
envio.to_csv(TRABAJO / "submission.csv")
print(f"submission.csv: {len(envio):,} filas", envio["row_type"].value_counts().to_dict())
envio.head()